# 1- Pretraitement

In [15]:
# importer les packages
import pandas as pd
import numpy as np

In [16]:
# lire le dataset
df = pd.read_csv("../data/raw/telco_customer_churn.csv")

In [17]:
# afficher le dataset
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [18]:
# info sur les donnees
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [19]:
# afficher la somme des doublons
df.duplicated().sum()

0

In [20]:
# afficher la somme des donnees nulles dans chaque variable
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [27]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
mask_na = df["TotalCharges"].isna()
print(f"{mask_na.sum()} valeurs non numériques converties en NaN")
display(df.loc[mask_na, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

11 valeurs non numériques converties en NaN


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,NaN,No
753,3115-CZMZD,0,20.25,NaN,No
936,5709-LVOEQ,0,80.85,NaN,No
1082,4367-NUYAO,0,25.75,NaN,No
1340,1371-DWPAZ,0,56.05,NaN,No
3331,7644-OMVMY,0,19.85,NaN,No
3826,3213-VVOLG,0,25.35,NaN,No
4380,2520-SGTTA,0,20.00,NaN,No
5218,2923-ARZLG,0,19.70,NaN,No
6670,4075-WKNIU,0,73.35,NaN,No


In [28]:
# Harmonisation des variables binaires & de la cible
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype("int8")

# Normalisation des libellés de modalités (snake-case léger sur les noms de colonnes)
df.columns = [c.strip() for c in df.columns]
print("Taux de churn global :", f"{df['Churn'].mean():.2%}")

Taux de churn global : 26.54%


In [ ]:
# Contrôles de cohérence numérique
checks = {
    "tenure dans [0, 72]": df["tenure"].between(0, 72).all(),
    "MonthlyCharges > 0": (df["MonthlyCharges"] > 0).all(),
    "TotalCharges >= 0": (df["TotalCharges"] >= 0).all(),
    "TotalCharges ≈ tenure × MonthlyCharges (écart médian < 10 %)":
        (((df["TotalCharges"] - df["tenure"] * df["MonthlyCharges"]).abs()
          / df["TotalCharges"].replace(0, np.nan)).median() < 0.10),
}
for k, v in checks.items(): print(("✅" if v else "❌"), k)

df = df.drop_duplicates(subset="customerID").reset_index(drop=True)